In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
      accuracy_score, precision_score, recall_score, f1_score,
      roc_auc_score, confusion_matrix, classification_report
)

## 1. EDA and basic data cleaning

In [2]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")


In [3]:
## clean unneccasry fields - customer id
df_cleaned = df.drop(columns='customerID')

In [4]:
## now the TotalCharges is str due to some missing values so first fixing that

df_cleaned['TotalCharges'] = pd.to_numeric(df_cleaned['TotalCharges'], errors='coerce').fillna(0)

In [5]:
df_cleaned['SeniorCitizen'] = df_cleaned['SeniorCitizen'].map({1:'Yes', 0: 'No'})
df_cleaned['Churn'] = df_cleaned['Churn'].map({'Yes':1,'No': 0})

## 2. Test train split

In [6]:
x = df_cleaned.drop(columns="Churn")
y = df_cleaned['Churn']

X_train, X_test, y_train, y_test = train_test_split(x,y,random_state=42,test_size=0.2,stratify=y)

## 3. Preprocessing

In [7]:
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
cat_cols = x.select_dtypes(include=['object','str']).columns

print(num_cols)
cat_cols

Index(['tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')


Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod'],
      dtype='str')

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore",drop="if_binary"),cat_cols),
        ("num","passthrough",num_cols)
    ]
)

In [9]:
pipline = Pipeline(
    steps=[
        ("preprocessing",preprocessor),
        # ("model",DecisionTreeClassifier(criterion="gini",random_state=42,max_depth=5))
        ("model",RandomForestClassifier(
                criterion="gini",random_state=42,n_estimators=100,max_depth=8
            )
        )   
    ]
)

### What this notebook fits

This notebook fits a Random Forest with `max_depth=8` on Telco churn.

Before landing on this, I tried three other versions: a default Decision Tree, a Decision Tree with `max_depth=5`, and a Random Forest with no depth cap. Each one had a problem worth understanding. The full story is in section 4 below.

In [10]:
pipline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [11]:
y_pred = pipline.predict(X_test)
y_pred

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))

In [12]:
correct = (y_pred == y_test).sum()
print(f"Correct : {correct}\nWrong : {y_test.size - correct }")

Correct : 1135
Wrong : 274


In [13]:
y_proba = pipline.predict_proba(X_test)[:, 1]                                       
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")     
print(f"Precision: {precision_score(y_test, y_pred):.4f}")                                             
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")                                                
print(f"F1:        {f1_score(y_test, y_pred):.4f}")                                                    
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")                                              
print("\nConfusion Matrix:")                               
print(confusion_matrix(y_test, y_pred))                                                                
print("\nClassification Report:")                          
print(classification_report(y_test, y_pred, target_names=['No-Churn', 'Churn']))  

Accuracy:  0.8055
Precision: 0.6724
Recall:    0.5214
F1:        0.5873
ROC-AUC:   0.8407

Confusion Matrix:
[[940  95]
 [179 195]]

Classification Report:
              precision    recall  f1-score   support

    No-Churn       0.84      0.91      0.87      1035
       Churn       0.67      0.52      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409



In [14]:
print(f"Train accuracy: {pipline.score(X_train, y_train):.4f}")                    
print(f"Test accuracy:  {pipline.score(X_test, y_test):.4f}")                      

Train accuracy: 0.8335
Test accuracy:  0.8055


## 4. What I learned

### Step 1: default Decision Tree overfits a lot

I started with `DecisionTreeClassifier(random_state=42)`.

| | Train acc | Test acc | Depth | Leaves |
|---|---|---|---|---|
| Default DT | 0.998 | 0.727 | 22 | 1101 |

Train and test are 27 points apart. The default tree has no depth limit. It keeps splitting until every leaf is pure, so it memorises the training data instead of learning the real pattern.

The numeric columns made it worse. A binary column like `gender` gives the tree only 1 possible split. But `tenure` (0–72) and the two charges columns are continuous — the tree has many thresholds to choose from, so many places to fit noise.

### Step 2: Decision Tree with max_depth=5

| | Train acc | Test acc | Depth | Leaves |
|---|---|---|---|---|
| DT (max_depth=5) | 0.804 | 0.798 | 5 | 32 |

The gap dropped from 27 points to 0.5 points. F1 = 0.599, ROC-AUC = 0.830.

Now compared to LR:

| Model | F1 | ROC-AUC |
|---|---|---|
| LR baseline (notebook 02) | 0.604 | 0.842 |
| DT (max_depth=5) | 0.599 | 0.830 |

Almost the same. I thought the tree would beat LR because trees can learn non-linear patterns. It didn't. That says something about the dataset, not the model.

Telco churn mostly comes from simple one-direction effects. Longer tenure means less churn. Month-to-month contract means more churn. Higher monthly charges means more churn. LR fits these patterns directly with one weight per feature. The tree does the same job using 32 leaves — same answer, more steps.

A tree would only beat LR if the data had effects LR can't draw a straight line for. For example "tenure matters only for fiber-optic customers" (an interaction), or "churn is high for both very low and very high charges" (non-monotone). Telco has a little of this, not enough.

### Step 3: Random Forest with max_depth=8

| | Train acc | Test acc |
|---|---|---|
| RF (max_depth=8, n_estimators=100) | 0.834 | 0.806 |

F1 = 0.587, ROC-AUC = 0.841, Precision = 0.672, Recall = 0.521.

ROC-AUC went up from DT (0.830 → 0.841). But F1 went down (0.599 → 0.587). I had to think about why.

ROC-AUC measures how well the model ranks positives above negatives across all possible thresholds. RF got better at ranking.

F1 is measured at one threshold (0.5). At 0.5, RF predicted positive less often than DT did — 290 vs 333 positive predictions. So precision went up (more sure when it says yes) and recall went down (it misses more real churners).

Why does RF predict positive less often at 0.5? Each of the 100 trees is built on a bootstrap sample where churners are still only ~25% of rows. When I average the probability outputs of 100 trees on a borderline customer, the average usually lands a bit below 0.5. The model is more cautious.

So even though RF ranks better, at the fixed 0.5 threshold its F1 didn't improve. Threshold tuning is on next week's plan — moving the threshold down should help.

### Step 4: I tried RF with no max_depth — it got worse

The usual advice is that RF doesn't need a depth limit because bootstrap + random feature subsets are supposed to handle variance. So I tested it.

| | Train acc | Test acc | F1 | ROC-AUC |
|---|---|---|---|---|
| RF (no max_depth) | 0.998 | 0.787 | 0.551 | 0.820 |
| RF (max_depth=8) | 0.834 | 0.806 | 0.587 | 0.841 |

Train 0.998 / test 0.787. Same problem I had with the default Decision Tree. So the cap helps even in RF on this dataset.

Why? On Telco only a few features (`tenure`, `Contract`, `MonthlyCharges`) carry most of the signal. Even when each tree gets a random subset of features, it still ends up picking these few in its top splits. So the trees make similar early decisions — they look more alike than the theory expects. When trees are similar, averaging them doesn't help much. The noise each tree picked up doesn't get cancelled out properly.

Capping depth at 8 forces each tree to stop before it memorises its bootstrap sample's noise. Less noise per tree → averaging actually works.

So "let RF trees grow fully" is a starting point, not a rule. On data like this, capping depth wins.

### Final comparison

| Model | F1 | ROC-AUC |
|---|---|---|
| LR baseline | 0.604 | 0.842 |
| DT (max_depth=5) | 0.599 | 0.830 |
| RF (no cap) | 0.551 | 0.820 |
| RF (max_depth=8) | 0.587 | 0.841 |

All the well-tuned models land near AUC 0.84. There's a ceiling on what this feature set can give me. To push past it I need class-imbalance handling, threshold tuning, or new features — not a fancier model. That's what next expirements will cover